<a href="https://colab.research.google.com/github/jmarrietar/LLMs-from-scratch-study/blob/feature%2Fstudy-session/Chapter_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn

In [ ]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your
     [0.55, 0.87, 0.66], # Journey
     [0.57, 0.85, 0.64], # starts
     [0.22, 0.58, 0.33], # with
     [0.77, 0.25, 0.10], # one
     [0.05, 0.80, 0.55]] # step
)

In [ ]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [ ]:
query = inputs[1]

In [ ]:
attn_scores_2 = torch.empty(inputs.shape[0])

for i, x_i in enumerate(inputs):
    attn_scores_2[i] = query.dot(x_i)

Matematicamente dot product es una forma de combinar dos vectores en un valor escalar.

Pero tambien lo podriamos ver como una "Medida" de "Similitud".

In [ ]:
attn_scores_2

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

Para normalizarlo:

In [ ]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

**Nota:** Los Attention "Weights", son los attention scores normalizados.

In [ ]:
print("Attention weights:", attn_scores_2)

Attention weights: tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [ ]:
print("Sum:", attn_weights_2_tmp.sum())

Sum: tensor(1.0000)


En la practica es mejor utilizar softmax para esto.

In [ ]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

In [ ]:
attn_weigths_2_naive = softmax_naive(attn_scores_2)

Ok, hasta aqui he calculado los pesos de attencion (Attention weigths):

In [ ]:
attn_weigths_2_naive

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

Ok Ahora si voy a calcular el vector de contexto (z) (El cual es como un vector con esteroides).

Ok va, Pero como lo calculo?

* El vector de **contexto Z(2)** es la suma ponderada de todos los input vectors, obtenida multiplicando todos los **`input vectors`** por su correspondiente **`peso de atencion`** (Attention Weigth).

In [ ]:
query = inputs[1]

In [ ]:
context_vec_2 = torch.zeros(query.shape)

In [ ]:
for i, x_i in enumerate(inputs):
    context_vec_2 = context_vec_2 + x_i*attn_weigths_2_naive[i]

Entonces, aqui lo que estamos haciendo es multiplicar, cada uno de los inputs por el peso de attencion correspondiente para la query y ese input.

Luego sumamos todo y ese seria el "vector de contexto"

In [ ]:
context_vec_2

tensor([0.4419, 0.6515, 0.5683])

### Computing Attention Weigths for all Input tokens

Obteniendolo multiplicando dos for `loops`

In [ ]:
attn_scores = torch.empty(6, 6)

In [ ]:
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = x_i.dot(x_j)

In [ ]:
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

Cada uno de los elementos del tensor representa el **score de atencion** entre cada uno de los pares de **inputs**.

Esto mismo se podria hacer de forma mas eficiente con Multiplicacion Matricial

#### Compute Attention Scores

In [ ]:
attn_scores = inputs @ inputs.T

In [ ]:
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

#### Compute Attention Weigths

In [ ]:
attn_weights = torch.softmax(attn_scores, dim=-1)

In [ ]:
attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

#### Compute Context Vectors

In [ ]:
all_context_vecs = attn_weights @ inputs

In [ ]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [ ]:
attn_weights[0]

tensor([0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452])

**Ojo:** Estos vectores en verdad representan es una palabra (o token)

In [ ]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

vectores con contexto

In [ ]:
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [ ]:
attn_weights[0] @ inputs

tensor([0.4421, 0.5931, 0.5790])

Al realizar la multiplicacion matricial, es mas optimo.

### 3.4 Implementing self-attention with trainable weights

El mecanismo de atencion utilizado en los modelos de GPT y en la arquitectura original se llama "**scaled dot-product attention**"

Existen estas matrices que son actualizadas durante el entrenamiento del modelo.

Entonces lo que nos ayudan a hacer estas matrices es proyectar el *embedding* de input token x_i en vectores *query* , *key* , *value*.

In [ ]:
x_2 = inputs[1]

In [ ]:
d_in = inputs.shape[1]
d_out = 2

In [ ]:
torch.manual_seed(123)

In [ ]:
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False) # Set to true when training
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [ ]:
W_query

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])

In [ ]:
W_key

Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])

In [ ]:
W_value

Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])

In [ ]:
# To get query, key, values:

query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

In [ ]:
key_2

tensor([0.4433, 1.1419])

In [ ]:
# Compute for all inputs
keys = inputs @ W_key
values = inputs @ W_value

In [ ]:
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]])

Entonces puedo sacar el score de atencion entre ese query_2 y el key_2 .

In [ ]:
# Attention score w_22
keys_2 = keys[1] # position 2
attn_score_22 = query_2.dot(keys_2) # Producto punto entre query_2 y key_2

In [ ]:
attn_score_22

tensor(1.8524)

Pero podria generalizarlo y sacar la atencion de ese query_2 con TODOS los keys!.

¿De que manera?

In [ ]:
# Attention score del query 2 con todos los demas Keys

attn_scores_2 = query_2 @ keys.T

In [ ]:
attn_scores_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])

Como podemos ver el segundo elemento en el output matchea el `attn_score` computado previamente.

* Ahora toca calcular el attention weigth a partir del attention score.

* Aunque esto es nomas escalarlo dividiendolo por la raiz quadrada del tamañana del embedding de las keys. Y aplicarles el softmax.

* Por que hacemos la escalada?.
  * A// La hacemos para mejorar el performance en el entrenamiento evitando gradientes muy pequeños.

In [ ]:
d_k = keys.shape[-1] # Dimension of keys

In [ ]:
attn_weigths_2 = torch.softmax(attn_scores_2 / (d_k**0.5), dim = -1)

In [ ]:
attn_weigths_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])

Ya tenemos los attention weigths, pero nos falta sacar el **vector de contexto** como una **suma ponderada** del vector de Valores (`Values`)

In [ ]:
values

tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]])

In [ ]:
context_vec_2 = attn_weigths_2 @ values # suma ponderada

Como yo lo veo es que hace una suma ponderada en cada una de las dimensiones y despues lo sumo tanto para la posicion 0 como para la 1.

In [ ]:
context_vec_2

tensor([0.3061, 0.8210])

Y me da como resultado un vector de contexto para esa Query.

#### 3.4.2 Implementing a compact self-attention Python Class

In [ ]:
import torch.nn as nn

In [ ]:
d_in = inputs.shape[1]
d_out = 2

In [ ]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        # Definimos las matrices que nos van a ayudar a proyectar (Q,K,V)
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        # Hacemos la proyeccion aqui
        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value

        attn_scores = queries @ keys.T

        d = keys.shape[-1]

        attn_weigth = torch.softmax(
            attn_scores/(d**0.5), dim=-1 # Aqui hago el escalamiento sobre dimension de las keys
        )

        context_vector = attn_weigth @ values # Lo que en verdad queremos

        return context_vector

In [ ]:
# Probemoslo

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


Se podria crear una v2 que utiliza nn.Linear en ves de nn.Parameter ya que tiene una inicializacion mas estable aparentemente.

In [ ]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        # Definimos las matrices que nos van a ayudar a proyectar (Q,K,V)
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        # Hacemos la proyeccion aqui
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T

        d = keys.shape[-1]

        attn_weigth = torch.softmax(
            attn_scores/(d**0.5), dim=-1 # Aqui hago el escalamiento sobre dimension de las keys
        )

        context_vector = attn_weigth @ values # Lo que en verdad queremos

        return context_vector

In [ ]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
sa_v2(inputs)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)

#### 3.5 Hiding Future words with causal attention

In [ ]:
queries = sa_v2.W_query(inputs) # Queries es lo que en mi imagen tengo como matriz_query (queries)
keys = sa_v2.W_key(inputs)

In [ ]:
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores/ keys.shape[-1]**0.5, dim=-1)

In [ ]:
attn_weights

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)

Note: torch.softmax(tensor, dim=-1) will apply the softmax function independently to each row.

Bueno y ahora como creamos la mascara ?
A// Utilizando la funcion tril de pytorch.

In [ ]:
attn_scores.shape

torch.Size([6, 6])

In [ ]:
context_length = attn_scores.shape[0]

In [ ]:
mask_simple = torch.tril(torch.ones(context_length, context_length))

In [ ]:
mask_simple

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

Ahora bien, esta mascara la podriamos multiplicar por el attention weigth y de esa forma solo tendriamos la atencion para los anteriores (los inputs anteriores).

In [ ]:
masked_simple = attn_weights*mask_simple

In [ ]:
masked_simple

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)

In [ ]:
# Renormalize the attention weigth to sum up to 1

row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums

In [ ]:
masked_simple_norm

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)

Otra manera de hacer la mascara:

Softmax: Recordando la funcion de softmax, para valores muyy negativos. Los pone cerca a cero.

Masking trick = Colocar 1 arriba de la diagonal y esos pasarlos -inf luego para que con el softmax se vayan a 0.

In [ ]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

In [ ]:
mask

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])

In [ ]:
mask.bool()

tensor([[False,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True],
        [False, False, False, False,  True,  True],
        [False, False, False, False, False,  True],
        [False, False, False, False, False, False]])

In [ ]:
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
masked

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)

In [ ]:
# Ahora si le aplicamos el softmax
attn_weigths = torch.softmax(masked / keys.shape[-1]**0.5, dim =-1)

#### 3.5.2 Masking additional attention weigths with dropout

In [ ]:
torch.manual_seed(123)

In [ ]:
dropout = torch.nn.Dropout(0.5)

In [ ]:
example = torch.ones(6,6)

In [ ]:
example

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [ ]:
dropout(example)

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])

Para compensar la reduccion de elementos, los elementos restantes son rescalados

In [ ]:
dropout(attn_weights)

tensor([[0.3843, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.3324, 0.0000, 0.3329, 0.2955],
        [0.0000, 0.3318, 0.3325, 0.2996, 0.3328, 0.2961],
        [0.0000, 0.0000, 0.3337, 0.3142, 0.0000, 0.3128],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3317, 0.3169],
        [0.3869, 0.3327, 0.0000, 0.3084, 0.3331, 0.3058]],
       grad_fn=<MulBackward0>)

#### 3.5.3 Implementing a compact causal attention class

In [ ]:
batch = torch.stack((inputs, inputs))

In [ ]:
batch

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        # Matrices - Proyeccion
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Attention score & weigths
        attn_scores = queries @ keys.transpose(1, 2)

        # Agregar la mascara de masked "CAUSAL attention"
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)

        # Escalamiento y Softmax
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1) # no olvidar el dim=-1 para aplicarlo a las rows

        # Aplicar Dropout
        attn_weigths = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec



In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

In [ ]:
context_vecs = ca(batch)

In [ ]:
context_vecs.shape

torch.Size([2, 6, 2])

In [ ]:
context_vecs[0,:,:]

tensor([[-0.4519,  0.2216],
        [-0.5874,  0.0058],
        [-0.6300, -0.0632],
        [-0.5675, -0.0843],
        [-0.5526, -0.0981],
        [-0.5299, -0.1081]], grad_fn=<SelectBackward0>)

In [ ]:
context_vecs[1,:,:]

tensor([[-0.4519,  0.2216],
        [-0.5874,  0.0058],
        [-0.6300, -0.0632],
        [-0.5675, -0.0843],
        [-0.5526, -0.0981],
        [-0.5299, -0.1081]], grad_fn=<SelectBackward0>)

## 3.6 Extending single-head attention to multi-head attention


##### 3.6.1 Stacking multiple single-head attention layers

Basciamente esto es crear Multiples INSTANCIAS del mecanismo de auto-atencion (self-attention).

IDEA CORE:
- Correr el mecanismo de atención Multiples veces, con diferentes proyecciones.

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
                for _ in range(num_heads)
            ]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [ ]:
torch.manual_seed(123)

In [ ]:
context_length = batch.shape[1]
context_length

6

In [ ]:
d_in, d_out = 3, 2

In [ ]:
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)

In [ ]:
mha.heads

ModuleList(
  (0-1): 2 x CausalAttention(
    (W_query): Linear(in_features=3, out_features=2, bias=False)
    (W_key): Linear(in_features=3, out_features=2, bias=False)
    (W_value): Linear(in_features=3, out_features=2, bias=False)
    (dropout): Dropout(p=0.0, inplace=False)
  )
)

In [ ]:
context_vecs = mha(batch)

In [ ]:
context_vecs

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)

##### 3.6.2 Implementing multi-head attention with weight splits

Ok la estrategia ahora es combinar las clases `MultiHeadAttentionWrapper` y `CausalAttention` en una sola.

In [ ]:

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads

        self.head_dim = d_out // num_heads  # -> Reduces de projection dum to match desired output dim

        # Seteamos las Matrices de pesos GRANDES
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Una Layer Linear para combinar el output de los heads.
        self.out_proj = nn.Linear(d_out, d_out)

        # Dropout
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
            )

    def forward(self, x):

        b, num_tokens, d_in = x.shape

        # La multiplicacion de los pesos Grandes con el Input
        # Esto nos da unas matrices (Q,K,V) grandes tambien

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Truco: Aca implicitamente SEPARO La matriz agregandole num_heads a la dimension
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)


        # Transpose from shape(b, num_tokens, num_heads, head_dim) to (b,num_heads, num_tokens,head_dim)
        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

        # Ahora: Todo como normalmente se estaba haciendo (Calcular attn scores)
        attn_scores = queries @ keys.transpose(2,3) # Dot product for each head
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # MASK: Aplicarle mascara para hacerlo Causal Attention
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # Attention weigth -> Apply softmax + escalamiento
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        # Aplicar dropout
        attn_weights = self.dropout(attn_weights)

        # Obtener Vector de Contexto (Aplicarlo ahora si )
        context_vec = (attn_weights @ values).transpose(1, 2)

        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out) # combinar heads

        # optional projection

        context_vec = self.out_proj(context_vec)

        return context_vec

In [ ]:
torch.manual_seed(123)

In [ ]:
batch_size, context_length, d_in = batch.shape

In [ ]:
d_out = 2

In [ ]:
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

In [ ]:
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
